In [1]:
from mitoCytoML import segmentor
from mitoCytoML import data
import os
import cv2
#from PIL import Image
#import matplotlib.pyplot as plt
#from stardist import random_label_cmap, _draw_polygons, export_imagej_rois
#lbl_cmap = random_label_cmap()
#%matplotlib inline

In [2]:
data_obj = data.data_obj ()

In [3]:
print(dir(data_obj))

['FR_masks_dir', 'FR_model_name', 'FR_removed_dir', 'NASM_classifier_name', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'create_dir_structure', 'final_segmented_masks_dir', 'get_FR_model_name', 'get_NASM_classifier_name', 'get_image_list', 'get_segmentation_model_name', 'individual_fibers_dir', 'mass_channel_dir', 'membrane_channel_dir', 'methods', 'model_FR_dir', 'model_classification_dir', 'model_segmentation_dir', 'raw_images_dir', 'raw_segmented_masks_dir', 'red_green_images_dir', 'segmentation_model_name']


In [4]:
data_obj.create_dir_structure()

In [5]:
membrane_list, mass_list = data_obj.get_image_list()

In [23]:
from keras_unet_collection import models, losses
from keras.models import load_model
import numpy as np
import cv2
from patchify import patchify, unpatchify
import math

def get_unet_model(name='FR_model.hdf5',directory= './model_FR/'):
    model = load_model(directory+name , custom_objects={"dice_coef": losses.dice_coef })
    return model

def unet_predict(image, patch_size=512):
    step_size= math.gcd(patch_size, image.shape[0],image.shape[1])
    #step_size=1
    model = get_unet_model()
    patches = patchify(image, (patch_size, patch_size,3), step=step_size) 
    predicted_patches = []
    print(step_size)
    for i in range(patches.shape[0]):
        for j in range(patches.shape[1]):
            single_patch = np.squeeze(patches[i,j,:,:])
            #single_patch_norm = np.expand_dims(normalize(np.array(single_patch), axis=1),2)
            print(single_patch.shape)
            
            
            single_patch_norm = np.array(single_patch)/255
            single_patch_input=np.expand_dims(single_patch_norm, 0)
            single_patch_prediction = (model.predict(single_patch_input)[0,:,:,0] > 0.5).astype(np.uint8)
            predicted_patches.append(single_patch_prediction)
            predicted_patches = np.array(predicted_patches)
            predicted_patches_reshaped = np.reshape(predicted_patches, (patches.shape[0], patches.shape[1], patch_size,patch_size,3) )
            segm_img = unpatchify(predicted_patches_reshaped, image.shape)
    
    return segm_img

In [24]:
for infile in membrane_list:
    infile_mass_channel_path= os.path.join(data_obj.mass_channel_dir, infile)
    infile_membrane_channel_path= os.path.join(data_obj.membrane_channel_dir, infile)
    outfile_path_red_green= os.path.join(data_obj.red_green_images_dir, infile)
    outfile_path_raw_fiber_segmentation= os.path.join(data_obj.raw_segmented_masks_dir, infile)
    outfile_path_folded_region= os.path.join(data_obj.FR_masks_dir, infile)
    mass_img= cv2.imread(infile_mass_channel_path,0)
    membrane_img= cv2.imread(infile_membrane_channel_path,0)
    
    segmentor_obj = segmentor.segmentor(membrane_img, mass_img)
    
    red_green_image = segmentor_obj.red_green_image_maker()
    cv2.imwrite(outfile_path_red_green,red_green_image)
    print(red_green_image.shape)
    
    #fiber_mask, details= segmentor_obj.fiber_segmentator()
    #cv2.imwrite(outfile_path_raw_fiber_segmentation,fiber_mask)
    
        
    #folded_region = segmentor_obj.folded_region_segmentor()
    #cv2.imwrite(outfile_path_raw_fiber_segmentation ,folded_region)
    
    folded_region = unet_predict(red_green_image)
    cv2.imwrite(outfile_path_raw_fiber_segmentation ,folded_region)
    
    
    
   

(2290, 1441, 3)
1
(512, 512, 3)
1/1 [==============================] - 2s 2s/step


ValueError: cannot reshape array of size 262144 into shape (1779,930,512,512,3)